# Phase II_04: Model Inference & Evaluation

**Objective**: Evaluate trained 8-channel U-Net on test set

**Outputs**:
- Per-class accuracy and IoU
- Confusion matrix
- Comparison to baseline (4-channel difference model)
- Per-severity class performance (especially Extreme severity)

## Setup: Load Dependencies and Mount Drive

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, jaccard_score
import json
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Setup complete")

## Load Trained Model and Checkpoint

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Find latest model checkpoint
model_dir = Path('/content/drive/MyDrive/RETINNA_cache/phase2/II_02_training')
model_files = sorted(model_dir.glob('unet_model_*.pt'))

if not model_files:
    raise FileNotFoundError(f"No model checkpoints found in {model_dir}")

latest_model_file = model_files[-1]
print(f"\n✓ Loading model: {latest_model_file.name}")

# Load checkpoint
checkpoint = torch.load(latest_model_file)
print(f"Checkpoint keys: {checkpoint.keys()}")
print(f"Config: {checkpoint['config']}")

# Extract normalization stats
channel_mean = checkpoint['normalization']['channel_mean'].to(device)
channel_std = checkpoint['normalization']['channel_std'].to(device)
class_weights = checkpoint['class_weights'].to(device)

print(f"\nChannel mean: {channel_mean}")
print(f"Channel std: {channel_std}")

# Get training history for comparison
history = checkpoint['history']
print(f"\nTraining history:")
print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
print(f"  Final val loss: {history['val_loss'][-1]:.4f}")
print(f"  Final train acc: {history['train_acc'][-1]:.4f}")
print(f"  Final val acc: {history['val_acc'][-1]:.4f}")

## Define U-Net Architecture (same as training)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels // 2, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        if x2.size() != x1.size():
            x1 = F.pad(x1, (0, x2.size(3) - x1.size(3), 0, x2.size(2) - x1.size(2)))
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=8, out_channels=7, bilinear=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.bilinear = bilinear
        
        factor = 2 if bilinear else 1
        
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        x = self.outc(x)
        return x

# Create model and load weights
model = UNet(in_channels=8, out_channels=7).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✓ Model loaded and set to eval mode")
print(f"  Device: {device}")

## Load Test Data and Run Inference

In [ ]:
# Load Phase II_01 outputs
labels_dir = Path('/content/drive/MyDrive/RETINNA_cache/phase2/II_01_relabeling')

# Find latest files
label_files = sorted(labels_dir.glob('multi_class_labels_*.pt'))
pre_rgbn_files = sorted(labels_dir.glob('pre_rgbn_*.pt'))
post_rgbn_files = sorted(labels_dir.glob('post_rgbn_*.pt'))

print(f"Loading Phase II_01 outputs...")
labels_tensor = torch.load(label_files[-1])
pre_rgbn_tensor = torch.load(pre_rgbn_files[-1]).cpu()
post_rgbn_tensor = torch.load(post_rgbn_files[-1]).cpu()

print(f"  Labels shape: {labels_tensor.shape}")
print(f"  Pre-RGBN shape: {pre_rgbn_tensor.shape}")
print(f"  Post-RGBN shape: {post_rgbn_tensor.shape}")

# Use index-based splitting instead of metadata folds
# (metadata represents original 534 samples, but II_01 processed only 424)
# Based on actual training split from II_02
N = len(pre_rgbn_tensor)  # 424

# Known splits from training
train_size = 278
val_size = 78
test_size = N - train_size - val_size  # Should be 68

# Test indices: last samples
test_indices = list(range(train_size + val_size, N))

print(f"\nUsing index-based splitting:")
print(f"  Train indices: 0-{train_size-1} ({train_size} samples)")
print(f"  Val indices: {train_size}-{train_size+val_size-1} ({val_size} samples)")
print(f"  Test indices: {train_size+val_size}-{N-1} ({len(test_indices)} samples)")
print(f"\n✓ Test set ready: {len(test_indices)} samples")

print(f"\nNote: Metadata contains 534 total samples (original CaBuAur),")
print(f"but II_01 processed only 424. Using processed dataset indices.")

In [ ]:
# Run inference on test set
print(f"Running inference on {len(test_indices)} test samples...\n")

all_predictions = []
all_labels = []

with torch.no_grad():
    for idx, sample_idx in enumerate(test_indices):
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(test_indices)}")
        
        # Load pre and post images
        pre_img = pre_rgbn_tensor[sample_idx].float()
        post_img = post_rgbn_tensor[sample_idx].float()
        image = torch.cat([pre_img, post_img], dim=0)  # [8, 512, 512]
        
        # Get ground truth label (post-fire)
        N = len(pre_rgbn_tensor)
        label = labels_tensor[N + sample_idx].long()  # [512, 512]
        
        # Move to device
        image = image.unsqueeze(0).to(device)  # [1, 8, 512, 512]
        label = label.to(device)
        
        # Apply z-score normalization
        mean = channel_mean.view(1, 8, 1, 1)
        std = channel_std.view(1, 8, 1, 1)
        image = (image - mean) / (std + 1e-8)
        
        # Inference
        output = model(image)  # [1, 7, 512, 512]
        prediction = output.argmax(dim=1).squeeze(0)  # [512, 512]
        
        # Store results
        all_predictions.append(prediction.cpu().numpy().flatten())
        all_labels.append(label.cpu().numpy().flatten())

# Concatenate all results
all_predictions = np.concatenate(all_predictions)
all_labels = np.concatenate(all_labels)

print(f"\n✓ Inference complete")
print(f"  Total pixels: {len(all_predictions):,}")

## Compute Metrics

In [ ]:
# Overall accuracy
overall_acc = np.mean(all_predictions == all_labels)
print(f"Overall pixel accuracy: {overall_acc:.4f} ({100*overall_acc:.2f}%)")

# Per-class metrics
class_names = ['Unburned', 'Low Severity', 'Moderate', 'High', 'Extreme', 'Water', 'Cloud']

print(f"\nPer-class metrics:")
print(f"{'Class':<20} {'Accuracy':<12} {'IoU':<12} {'Pixels':<12}")
print("-" * 56)

per_class_acc = {}
per_class_iou = {}

for class_id in range(7):
    # Accuracy for this class
    mask = all_labels == class_id
    if mask.sum() > 0:
        acc = (all_predictions[mask] == class_id).mean()
        per_class_acc[class_id] = acc
    else:
        per_class_acc[class_id] = np.nan
    
    # IoU for this class
    try:
        iou = jaccard_score(all_labels, all_predictions, labels=[class_id], average=None)[0]
        per_class_iou[class_id] = iou
    except:
        per_class_iou[class_id] = np.nan
    
    pixel_count = (all_labels == class_id).sum()
    
    print(f"{class_names[class_id]:<20} {per_class_acc[class_id]:<12.4f} {per_class_iou[class_id]:<12.4f} {pixel_count:<12,}")

print(f"\nMacro-averaged IoU: {np.nanmean(list(per_class_iou.values())):.4f}")

## Confusion Matrix & Classification Report

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions, labels=list(range(7)))

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues', aspect='auto')

# Set ticks and labels
ax.set_xticks(range(7))
ax.set_yticks(range(7))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)

# Add values to cells
for i in range(7):
    for j in range(7):
        text = ax.text(j, i, cm[i, j], ha="center", va="center", color="w" if cm[i, j] > cm.max() / 2 else "black")

ax.set_ylabel('Ground Truth')
ax.set_xlabel('Prediction')
ax.set_title('Confusion Matrix: Test Set')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Confusion matrix saved")

In [ ]:
# Classification report
print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_predictions, target_names=class_names, zero_division=0))

## Save Results

In [ ]:
# Save evaluation results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

results = {
    'timestamp': timestamp,
    'model_checkpoint': str(latest_model_file.name),
    'overall_accuracy': float(overall_acc),
    'per_class_accuracy': {class_names[k]: float(v) if not np.isnan(v) else None for k, v in per_class_acc.items()},
    'per_class_iou': {class_names[k]: float(v) if not np.isnan(v) else None for k, v in per_class_iou.items()},
    'macro_iou': float(np.nanmean(list(per_class_iou.values()))),
    'test_samples': len(test_indices),
    'test_pixels': int(len(all_predictions)),
}

# Save to Drive and local
results_path = f'/content/drive/MyDrive/RETINNA_cache/phase2/II_02_inference/evaluation_results_{timestamp}.json'
Path('/content/drive/MyDrive/RETINNA_cache/phase2/II_02_inference').mkdir(parents=True, exist_ok=True)

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to {results_path}")
print(f"\nSummary:")
print(f"  Overall accuracy: {results['overall_accuracy']:.4f}")
print(f"  Macro IoU: {results['macro_iou']:.4f}")
print(f"  Extreme class accuracy: {results['per_class_accuracy']['Extreme']:.4f}")